# Bank of Canada Rate Decisions — Your Starter Agent

**If you're not sure what to do next, continue from here.**

This notebook is a fresh, hackable agent for the BoC rate-decision use case — deliberately *not* wired into the numbered curriculum. It gives you our common building blocks behind simple toggles, so you can start building something of your own:

- **optional news search** — bounded, cutoff-aware Google Search (proxy-only)
- **optional code execution** — an E2B Python sandbox
- **two lightweight skills** — *tool-usage playbooks* in `starter_agent/skills/`

It does two things: lets you **talk to the agent** (open-ended, Track 2) and **score one real forecast** (Track 1). The live cells are gated by `RUN_AGENT` so a fresh `Run All` is safe and free; flip it to `True` to actually call the model.


In [12]:
import warnings
from pathlib import Path


warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv


# Repo root holds the .env with PROXY_* creds the agent needs.
ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env", override=False)

# ── Model selection ───────────────────────────────────
# Two project models: "gemini-3.1-flash-lite-preview" (lite/default) and
# "gemini-3.5-flash" (advanced). Lite is the default.
AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # advanced (higher cost/latency)

# ── Capability toggles ────────────────────────────────
ENABLE_SEARCH = True
ENABLE_CODE_EXEC = True  # needs E2B_API_KEY
STYLE = "balanced"  # balanced | skeptical | cautious

# ── Spend guard ─────────────────────────────────────
RUN_AGENT = True  # set True only when you want live model calls

from boc_rate_decisions.starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)


print(
    "RUN_AGENT =", RUN_AGENT,
    "| model =", AGENT_MODEL,
    "| search =", ENABLE_SEARCH,
    "| code_exec =", ENABLE_CODE_EXEC,
    "| style =", STYLE,
)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview | search = True | code_exec = True | style = balanced


---
## 1. Meet your agent

`build_starter_agent_config` returns an `AgentConfig` with two toggles. The default turns **news search on** (proxy-only, no extra key) and **code execution off** (it needs `E2B_API_KEY` and is slower). Flip them and re-run — the loaded skills follow the enabled tools.


In [3]:
from importlib import reload

import boc_rate_decisions.starter_agent.agent as starter_agent_impl

starter_agent_impl = reload(starter_agent_impl)


def build_styled_config(
    *,
    enable_search: bool = ENABLE_SEARCH,
    enable_code_exec: bool = ENABLE_CODE_EXEC,
    style: str = STYLE,
):
    return starter_agent_impl.build_starter_agent_config(
        model=AGENT_MODEL,
        enable_search=enable_search,
        enable_code_exec=enable_code_exec,
        style=style,
    )


config = build_styled_config()

print("Agent:", config.name)
print("Search enabled:    ", config.context_retrieval.enabled)
print("Code-exec enabled: ", config.code_execution.enabled)
print("Style selected:    ", STYLE)
print("Skills loaded:     ", [p.name for p in config.skills_dirs])

preview_chars = 1800
full_instruction = config.instruction
start = max(0, len(full_instruction) - preview_chars)
if start > 0:
    next_newline = full_instruction.find("\n", start)
    if next_newline != -1 and next_newline + 1 < len(full_instruction):
        start = next_newline + 1

print("\n── System instruction preview (tail) ──\n")
print(full_instruction[start:])

Agent: boc_starter_agent
Search enabled:     True
Code-exec enabled:  True
Style selected:     balanced
Skills loaded:      ['forecasting', 'research-playbook', 'code-analysis-playbook']

── System instruction preview (tail) ──

## Role

You are a Bank of Canada monetary-policy analyst — fluent in the policy-rate path, the Bank's inflation-targeting framework, labour-market and bond-market conditions, and the Bank's institutional behaviour (gradualism, data dependence, reluctance to surprise markets). This is a starter agent: keep your reasoning transparent and your claims honest.

## How to respond

- For open-ended questions, scenario analysis, or anything conversational, answer directly and concisely — do NOT ask for a JSON payload.
- When you are handed a task that asks for a structured probability distribution over the next decision, produce a calibrated one.

## Analysis discipline

- Use `as_of` as a hard cutoff. If evidence is missing, say so explicitly.
- Start from historical

---
## Compare rationale with and without code execution

This is the concrete version of the first `Make it yours` exercise: hold the question, origin, model, and style fixed, then compare the forecast rationale with `enable_code_exec=False` versus `enable_code_exec=True`.

Keep `COMPARE_CODE_EXEC=False` unless you want to spend tokens and, for the code-exec path, E2B credits.

In [4]:
COMPARE_CODE_EXEC = True
COMPARE_STYLE = STYLE
COMPARE_ORIGIN_MODE = "latest_resolved"  # latest_resolved | manual
COMPARE_MANUAL_ANNOUNCEMENT = "2025-06-04"
COMPARE_LEAD_DAYS = 28


def _resolve_compare_origin(dir_df: pd.DataFrame, mode: str, manual_announcement: str):
    if mode == "latest_resolved":
        selected = pd.Timestamp(dir_df.iloc[-1]["timestamp"]).normalize()
        realized = {-1.0: "cut", 0.0: "hold", 1.0: "hike"}[float(dir_df.iloc[-1]["value"])]
        return selected, realized
    if mode == "manual":
        selected = pd.Timestamp(manual_announcement).normalize()
        hit = dir_df.loc[pd.to_datetime(dir_df["timestamp"]).dt.normalize() == selected]
        if hit.empty:
            raise ValueError(
                f"COMPARE_MANUAL_ANNOUNCEMENT={manual_announcement} not found in realized direction history."
            )
        realized = {-1.0: "cut", 0.0: "hold", 1.0: "hike"}[float(hit.iloc[-1]["value"])]
        return selected, realized
    raise ValueError("COMPARE_ORIGIN_MODE must be one of: latest_resolved, manual")


def _comparison_verdict(probs: dict[str, float]) -> str:
    ordered = sorted(probs.items(), key=lambda item: item[1], reverse=True)
    top_label, top_prob = ordered[0]
    runner_label, runner_prob = ordered[1]
    return f"{top_label.upper()} ({top_prob:.0%}) over {runner_label.upper()} ({runner_prob:.0%})"


if RUN_AGENT and COMPARE_CODE_EXEC:
    from datetime import datetime, timezone
    from aieng.forecasting.evaluation.task import ForecastingTask
    from boc_rate_decisions.data import (
        DIRECTION_SERIES_ID,
        DIRECTION_TASK_CATEGORIES,
        build_boc_service,
    )

    svc = build_boc_service(
        statcan_cache_dir=ROOT / "data" / "statcan",
        fred_cache_dir=ROOT / "data" / "fred",
    )
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    dir_df = svc.get_series(DIRECTION_SERIES_ID, as_of=now).copy()
    selected_announcement, realized = _resolve_compare_origin(
        dir_df,
        COMPARE_ORIGIN_MODE,
        COMPARE_MANUAL_ANNOUNCEMENT,
    )
    as_of = selected_announcement - pd.Timedelta(days=COMPARE_LEAD_DAYS)

    task = ForecastingTask(
        task_id="boc_code_exec_comparison",
        target_series_id=DIRECTION_SERIES_ID,
        horizons=[COMPARE_LEAD_DAYS],
        frequency="D",
        payload_type="categorical",
        categories=DIRECTION_TASK_CATEGORIES,
        description="BoC rate decision direction (cut/hold/hike), 28 days ahead (code-exec comparison).",
    )
    ctx = svc.context(as_of=as_of)

    variants = [
        ("search_only", build_styled_config(enable_code_exec=False, style=COMPARE_STYLE)),
        ("search_plus_code", build_styled_config(enable_code_exec=True, style=COMPARE_STYLE)),
    ]

    print(
        f"Comparing rationale at decision={selected_announcement.date()} from as_of={as_of.date()} "
        f"(actual={realized.upper()})"
    )

    for label, cfg in variants:
        pred = build_starter_agent_predictor(cfg).predict(task, ctx)[0]
        probs = pred.payload.probabilities
        rationale = pred.metadata.get("rationale", "")
        key_signals = pred.metadata.get("key_signals", [])
        print(f"\n== {label} ==")
        print("skills:", [p.name for p in cfg.skills_dirs])
        print("verdict:", _comparison_verdict(probs))
        print(f"p(actual={realized}) = {probs[realized]:.2%}")
        print("key_signals:", key_signals if key_signals else "[none returned]")
        print("rationale preview:")
        print(rationale[:900] if rationale else "[no rationale returned]")
else:
    print(
        "Set RUN_AGENT=True and COMPARE_CODE_EXEC=True to compare rationale with and without code execution."
    )

Comparing rationale at decision=2026-07-15 from as_of=2026-06-17 (actual=HOLD)

== search_only ==
skills: ['forecasting', 'research-playbook']
verdict: HOLD (50%) over CUT (45%)
p(actual=hold) = 50.00%
key_signals: ['Inflation gap remains positive at 0.82%, keeping the Bank cautious about easing prematurely.', 'Unemployment momentum is negative (-0.40%), signaling labor market cooling that invites policy support.', 'Yield spread at 0.5% provides no clear signal for a hawkish pivot or emergency easing.']
rationale preview:
As of June 17, 2026, the BoC is in a 'hold' phase at 2.25%. The primary driver for potential easing is the positive inflation gap (0.82%), suggesting persistent price pressure above target, countered by the negative unemployment momentum (-0.40%), which indicates softening labour market conditions. The yield spread of 0.5% is moderately positive, not signaling immediate recessionary alarm but reflecting a normalized curve. Given the Bank's strong history of 'hold' bia

---
## Talk to it  *(open-ended analysis)*

Ask the agent anything. This is the interactive mode: no scoring, no schema — just reasoning (and a web search, since search is on). Edit the question and explore.


In [6]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig
import logging


QUESTION_PRESETS = {
    "base": (
        "What is the case for a cut versus a hold at the Bank of Canada's next "
        "rate decision, and which looks more likely? Keep it concise."
    ),
    "hawkish_risk": (
        "What would need to happen over the next 4-8 weeks for a hike to become "
        "plausible despite current expectations? Provide trigger conditions."
    ),
    "dovish_risk": (
        "What are the strongest downside-growth or disinflation signals that could "
        "force a cut sooner than markets expect?"
    ),
}

QUESTION_KEY = "base"
QUESTION = QUESTION_PRESETS[QUESTION_KEY]


async def _run_text_without_cancel_noise(runner: AdkTextRunner, prompt: str) -> str:
    class _CancelFilter(logging.Filter):
        def filter(self, record: logging.LogRecord) -> bool:
            message = record.getMessage()
            return message != "Root node boc_starter_agent was cancelled."

    logger = logging.getLogger("google_adk.google.adk.runners")
    cancel_filter = _CancelFilter()
    logger.addFilter(cancel_filter)
    try:
        reply = await runner.run_text_async(prompt)  # noqa: F704, PLE1142
    finally:
        logger.removeFilter(cancel_filter)
    return reply

if RUN_AGENT:
    chat_agent = build_adk_agent(config)  # schema-free: plain text in, text out
    runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="boc_starter_chat"))
    reply = await _run_text_without_cancel_noise(runner, QUESTION)
    print(reply)
else:
    print("RUN_AGENT is False — set it to True in the setup cell to talk to the agent.")
    print(f"Current QUESTION_KEY={QUESTION_KEY!r}. Available: {list(QUESTION_PRESETS)}")

As of August 6, 2026, the Bank of Canada (BoC) is in a "wait-and-see" regime with the overnight rate at 2.25%. 

### Case for a Cut
*   **Economic Slack:** If the Governing Council assesses that the cumulative effect of past policy tightening—coupled with recent global trade headwinds—is creating more downside risk to growth than anticipated, they may cut to provide further insurance.
*   **Output Gap:** If data confirms the economy is operating with significant spare capacity, a cut would be the logical move to prevent inflation from undershooting the 2% target.

### Case for a Hold (Current Status)
*   **Data Dependence:** Inflation remains in a delicate position, with underlying measures occasionally showing persistence around 2.5% and energy-price volatility providing upward pressure. The BoC has signaled a preference for evidence that inflation is sustainably on the 2% path before easing further.
*   **Labour Market Resilience:** Recent signs of tightening or stability in the labo

Root node boc_starter_agent was cancelled.


In [9]:
# Optional verification: does the agent appear to pick up the research-playbook query pack?
# Keep False by default to avoid extra live token spend.
RUN_SKILL_VERIFICATION = True

from pathlib import Path
import re

SKILL_PATH = (
    ROOT
    / "implementations"
    / "boc_rate_decisions"
    / "starter_agent"
    / "skills"
    / "research-playbook"
    / "SKILL.md"
)

expected_topics = [
    "statement speech monetary policy report",
    "cpi core inflation",
    "employment unemployment wage growth",
    "overnight index swaps economist survey",
    "oil prices loonie exchange rate us tariffs",
]

print(f"Skill file: {SKILL_PATH}")
print("Skill exists:", SKILL_PATH.exists())
if SKILL_PATH.exists():
    skill_text = SKILL_PATH.read_text(encoding="utf-8").lower()
    found_topics = [topic for topic in expected_topics if topic in skill_text]
    print(f"Skill topic markers found: {len(found_topics)}/{len(expected_topics)}")
    for topic in found_topics:
        print("  -", topic)

if RUN_AGENT and RUN_SKILL_VERIFICATION:
    if "runner" not in globals() or "chat_agent" not in globals():
        chat_agent = build_adk_agent(config)
        runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="boc_skill_check"))

    verification_prompt = (
        "Tool-use verification check. If the research-playbook skill is available, start with the exact line "
        "'SKILL_USED: research-playbook'. Then output exactly five lines in this format: "
        "search_web(query=..., cutoff_date=<as_of>). The five lines must cover: "
        "(1) BoC statements/speeches/MPR, (2) CPI/core inflation, (3) labour market, "
        "(4) OIS/economist surveys, (5) oil/loonie/US tariffs/trade shock. "
        "Do not provide final macro analysis; only the five query lines."
    )

    verification_reply = await _run_text_without_cancel_noise(runner, verification_prompt)
    print("\n--- Verification reply ---\n")
    print(verification_reply)

    text = verification_reply.lower()
    query_lines = re.findall(r"search_web\(", text)
    checks = {
        "declares_skill_header": "skill_used: research-playbook" in text,
        "mentions_cutoff_date": "cutoff_date" in text,
        "contains_5_query_lines": len(query_lines) == 5,
        "covers_boc_comms": any(k in text for k in ["statement", "speech", "monetary policy report"]),
        "covers_inflation": any(k in text for k in ["cpi", "core inflation", "cpi-trim", "cpi-median"]),
        "covers_labour": any(k in text for k in ["unemployment", "employment", "labour"]),
        "covers_market_pricing": any(k in text for k in ["ois", "overnight index swaps", "economist survey"]),
        "covers_external_shocks": any(k in text for k in ["oil", "loonie", "exchange rate", "tariff", "trade"]),
    }

    print("\n--- Heuristic verification checks ---")
    for name, ok in checks.items():
        print(f"{name}: {'PASS' if ok else 'FAIL'}")

    print(
        "\nInterpretation: PASS signals make skill pickup likely, but strict proof is still the tool-call trace "
        "(e.g., Langfuse showing actual search_web arguments)."
    )
else:
    print(
        "\nSet RUN_AGENT=True and RUN_SKILL_VERIFICATION=True to run the live pickup verification probe."
    )

Skill file: /home/coder/agentic-forecasting-slf/implementations/boc_rate_decisions/starter_agent/skills/research-playbook/SKILL.md
Skill exists: True
Skill topic markers found: 5/5
  - statement speech monetary policy report
  - cpi core inflation
  - employment unemployment wage growth
  - overnight index swaps economist survey
  - oil prices loonie exchange rate us tariffs

--- Verification reply ---

SKILL_USED: research-playbook
search_web(query="Bank of Canada statement speech Monetary Policy Report overnight rate outlook", cutoff_date="2025-05-15")
search_web(query="Canada CPI core inflation CPI-trim CPI-median latest print", cutoff_date="2025-05-15")
search_web(query="Canada employment unemployment wage growth labour force survey latest", cutoff_date="2025-05-15")
search_web(query="Bank of Canada overnight index swaps economist survey next meeting", cutoff_date="2025-05-15")
search_web(query="Canada oil prices loonie exchange rate US tariffs trade shock growth outlook", cutoff_d

Root node boc_starter_agent was cancelled.


---
## 3. Score one prediction against a known outcome 

Run the agent as a `Predictor` and compare its probabilities with the realized BoC decision. You can now choose the forecast origin mode:
- `latest_resolved`: most recent already-resolved decision (default)
- `manual`: pick a specific announcement date from history
- `upcoming`: next scheduled meeting (no realized outcome yet; live forecasting mode)

Live, so this section is gated by `RUN_AGENT`.

In [10]:
from datetime import datetime, timezone


def _reasoning_audit(reasoning: str) -> dict[str, object]:
    text = (reasoning or "").lower()
    checks = {
        "mentions_cutoff_or_timing": any(k in text for k in ["as_of", "cutoff", "before", "after"]),
        "mentions_base_rates_or_history": any(
            k in text for k in ["base rate", "histor", "cycle", "previous decision", "meeting history"]
        ),
        "mentions_scenarios_or_alternatives": any(
            k in text for k in ["scenario", "alternative", "if", "risk", "tail", "upside", "downside"]
        ),
        "mentions_key_macro_signals": any(
            k in text
            for k in [
                "inflation",
                "cpi",
                "unemployment",
                "labour",
                "yield",
                "bond",
                "oil",
                "exchange rate",
            ]
        ),
    }
    score = int(sum(checks.values()))
    return {"score": score, "max_score": len(checks), "checks": checks}


def _policy_verdict(probs: dict[str, float]) -> str:
    ordered = sorted(probs.items(), key=lambda item: item[1], reverse=True)
    top_label, top_prob = ordered[0]
    runner_label, runner_prob = ordered[1]
    margin = top_prob - runner_prob
    if top_prob >= 0.7 and margin >= 0.2:
        strength = "high conviction"
    elif top_prob >= 0.55 and margin >= 0.1:
        strength = "moderate conviction"
    else:
        strength = "low conviction"
    return (
        f"Verdict: {top_label.upper()} ({top_prob:.0%}) over {runner_label.upper()} "
        f"({runner_prob:.0%}); margin={margin:.0%} [{strength}]."
    )


if RUN_AGENT:
    from aieng.forecasting.evaluation.task import ForecastingTask
    from aieng.forecasting.methods import CategoricalFrequencyPredictor
    from boc_rate_decisions.data import (
        DIRECTION_SERIES_ID,
        DIRECTION_TASK_CATEGORIES,
        build_boc_service,
    )

    ORIGIN_MODE = "latest_resolved"  # latest_resolved | manual | upcoming
    MANUAL_ANNOUNCEMENT = "2025-06-04"  # used only when ORIGIN_MODE == "manual"
    LEAD_DAYS = 28

    svc = build_boc_service(
        statcan_cache_dir=ROOT / "data" / "statcan",
        fred_cache_dir=ROOT / "data" / "fred",
    )
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    dir_df = svc.get_series(DIRECTION_SERIES_ID, as_of=now).copy()

    if ORIGIN_MODE == "latest_resolved":
        selected_announcement = pd.Timestamp(dir_df.iloc[-1]["timestamp"]).normalize()
        realized = {-1.0: "cut", 0.0: "hold", 1.0: "hike"}[float(dir_df.iloc[-1]["value"])]
    elif ORIGIN_MODE == "manual":
        selected_announcement = pd.Timestamp(MANUAL_ANNOUNCEMENT).normalize()
        hit = dir_df.loc[pd.to_datetime(dir_df["timestamp"]).dt.normalize() == selected_announcement]
        if hit.empty:
            raise ValueError(
                f"MANUAL_ANNOUNCEMENT={MANUAL_ANNOUNCEMENT} not found in realized direction history."
            )
        realized = {-1.0: "cut", 0.0: "hold", 1.0: "hike"}[float(hit.iloc[-1]["value"])]
    elif ORIGIN_MODE == "upcoming":
        schedule_path = ROOT / "implementations" / "boc_rate_decisions" / "meeting_schedule.yaml"
        if not schedule_path.exists():
            raise FileNotFoundError(f"Missing meeting schedule: {schedule_path}")
        import yaml

        with schedule_path.open("r", encoding="utf-8") as f:
            payload = yaml.safe_load(f)
        fixed_dates = sorted(pd.Timestamp(x).normalize() for x in payload.get("fixed_announcement_dates", []))
        upcoming = [d for d in fixed_dates if d > pd.Timestamp(now).normalize()]
        if not upcoming:
            raise ValueError("No upcoming announcement date found in meeting_schedule.yaml")
        selected_announcement = upcoming[0]
        realized = None
    else:
        raise ValueError("ORIGIN_MODE must be one of: latest_resolved, manual, upcoming")

    AS_OF = selected_announcement - pd.Timedelta(days=LEAD_DAYS)

    task = ForecastingTask(
        task_id="boc_starter_direction",
        target_series_id=DIRECTION_SERIES_ID,
        horizons=[LEAD_DAYS],
        frequency="D",
        payload_type="categorical",
        categories=DIRECTION_TASK_CATEGORIES,
        description="BoC rate decision direction (cut/hold/hike), 28 days ahead (starter).",
    )
    ctx = svc.context(as_of=AS_OF)
    pred = build_starter_agent_predictor(config).predict(task, ctx)[0]
    floor = CategoricalFrequencyPredictor().predict(task, ctx)[0]

    probs = pred.payload.probabilities
    print(
        f"Mode={ORIGIN_MODE} | decision={selected_announcement.date()} | "
        f"forecast_origin={AS_OF.date()} (T-{LEAD_DAYS})"
    )
    print(_policy_verdict(probs))
    if realized is not None:
        print(f"Actual outcome: {realized.upper()}\n")
    else:
        print("Actual outcome: N/A (upcoming decision)\n")

    print("  outcome   agent prob   climatology")
    for label in ("cut", "hold", "hike"):
        mark = ""
        if realized is not None and label == realized:
            mark = "   <- ACTUAL"
        print(f"  {label:<7}   {probs[label]:7.2%}     {floor.payload.probabilities[label]:7.2%}{mark}")

    if realized is not None:
        top = max(probs, key=probs.get)
        print(
            f"\nAgent put {probs[realized]:.0%} on what happened "
            f"({'its top pick ✓' if top == realized else f'top pick was {top}'})."
        )

    reasoning = pred.metadata.get("reasoning", "")
    if reasoning:
        audit = _reasoning_audit(reasoning)
        print(f"\nReasoning quality score: {audit['score']}/{audit['max_score']}")
        for name, passed in audit["checks"].items():
            print(f"  - {name}: {'yes' if passed else 'no'}")
        print("\nReasoning preview:", reasoning[:400])
else:
    print("RUN_AGENT is False — set it to True to score a live forecast.")

Mode=latest_resolved | decision=2026-07-15 | forecast_origin=2026-06-17 (T-28)
Verdict: HOLD (70%) over CUT (25%); margin=45% [high conviction].
Actual outcome: HOLD

  outcome   agent prob   climatology
  cut        25.00%      10.71%
  hold       70.00%      76.43%   <- ACTUAL
  hike        5.00%      12.86%

Agent put 70% on what happened (its top pick ✓).


# END of the notebook